In [2]:
# ============================================================
# CELL 1
# IMPORTS
# ============================================================

import os
import json
import random
import time

import numpy as np
import pandas as pd
from PIL import Image
import onnxruntime as ort

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [3]:
# ============================================================
# CELL 2
# DATASET SETUP
# ============================================================

DATASET_ROOT = os.path.join(r"c:\Users\priya\Downloads\ok\TinyBayes-ML", "data", "dataset", "cocoa")

SAVE_DIR = os.path.join(
    DATASET_ROOT,
    "run"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

RANDOM_STATE = 42

IMAGE_SIZE = 224

CLASS_NAMES = ['anthracnose', 'cssvd', 'healthy']

print("="*60)
print("CLASSES")
print("="*60)

print(CLASS_NAMES)

# ------------------------------------------------------------
# Build dataframe from dataset directory
# ------------------------------------------------------------

data = []
for cls_name in CLASS_NAMES:
    cls_dir = os.path.join(DATASET_ROOT, cls_name)
    if os.path.exists(cls_dir):
        for img_file in os.listdir(cls_dir):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                data.append({'Image_ID': img_file, 'class': cls_name})

df = pd.DataFrame(data)

print()
print("Total Images :", len(df))
print()
print(df["class"].value_counts())

# ------------------------------------------------------------
# Encode labels
# ------------------------------------------------------------

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["class"])
NUM_CLASSES = len(label_encoder.classes_)

print()
print("Label Mapping")
for i, cls in enumerate(label_encoder.classes_):
    print(i, "->", cls)

# ------------------------------------------------------------
# Train / Validation Split (Stratified 80 / 20)
# ------------------------------------------------------------

df_train, df_val = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=RANDOM_STATE
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

print()
print("="*60)
print("TRAIN SPLIT")
print("="*60)
print(df_train["class"].value_counts())

print()
print("="*60)
print("VALIDATION SPLIT")
print("="*60)
print(df_val["class"].value_counts())

# ------------------------------------------------------------
# Save splits
# ------------------------------------------------------------

df_train.to_csv(os.path.join(SAVE_DIR, "train_split.csv"), index=False)
df_val.to_csv(os.path.join(SAVE_DIR, "validation_split.csv"), index=False)

print()
print("Saved")
print(os.path.join(SAVE_DIR, "train_split.csv"))
print(os.path.join(SAVE_DIR, "validation_split.csv"))


CLASSES
['anthracnose', 'cssvd', 'healthy']

Total Images : 5523

class
cssvd          2237
healthy        1720
anthracnose    1566
Name: count, dtype: int64

Label Mapping
0 -> anthracnose
1 -> cssvd
2 -> healthy

TRAIN SPLIT
class
cssvd          1789
healthy        1376
anthracnose    1253
Name: count, dtype: int64

VALIDATION SPLIT
class
cssvd          448
healthy        344
anthracnose    313
Name: count, dtype: int64

Saved
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\cocoa\run\train_split.csv
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\cocoa\run\validation_split.csv


In [4]:
train_ids = set(df_train["Image_ID"])
val_ids = set(df_val["Image_ID"])

overlap = train_ids.intersection(val_ids)

print("Overlap:", len(overlap))


Overlap: 0


In [5]:
# ============================================================
# SAVE CLASS NAMES
# ============================================================

CLASS_NAMES_PATH = os.path.join(
    SAVE_DIR,
    "class_names.json"
)

with open(CLASS_NAMES_PATH, "w") as f:
    json.dump(CLASS_NAMES, f, indent=4)

print("Class names saved to:")
print(CLASS_NAMES_PATH)

print()
print("Classes:")
print(CLASS_NAMES)


Class names saved to:
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\cocoa\run\class_names.json

Classes:
['anthracnose', 'cssvd', 'healthy']


In [6]:
# ============================================================
# CELL 3
# EXTRACT MOBILENET FEATURES (LOCAL ONNX BACKBONE)
# ============================================================

TRAIN_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "train_features.csv"
)

VAL_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "validation_features.csv"
)

if os.path.exists(TRAIN_FEATURE_PATH) and os.path.exists(VAL_FEATURE_PATH):
    print("=" * 60)
    print("FEATURE FILES FOUND - LOADING FROM CACHE")
    print("=" * 60)
    train_features_df = pd.read_csv(TRAIN_FEATURE_PATH)
    val_features_df = pd.read_csv(VAL_FEATURE_PATH)
    print("Train features:", train_features_df.shape)
    print("Val features  :", val_features_df.shape)
else:
    print("=" * 60)
    print("EXTRACTING FEATURES VIA LOCAL MOBILENETV3 ONNX BACKBONE")
    print("=" * 60)
    ONNX_CANDIDATES = [
        os.path.abspath(os.path.join(DATASET_ROOT, "..", "..", "..", "..", "TinyBayes-App", "app", "src", "main", "assets", "models", "mobilenet_v3_small_features.onnx")),
        os.path.abspath(os.path.join(DATASET_ROOT, "..", "..", "models", "mobilenet_v3_small_features.onnx")),
    ]
    ONNX_PATH = next((p for p in ONNX_CANDIDATES if os.path.exists(p)), None)
    if not ONNX_PATH:
        raise FileNotFoundError(f"Could not find mobilenet_v3_small_features.onnx in candidates: {ONNX_CANDIDATES}")

    print(f"Loading ONNX Backbone: {ONNX_PATH}")
    session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
    inp_name = session.get_inputs()[0].name
    out_name = session.get_outputs()[0].name

    MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def extract_features(df_split):
        rows = []
        total = len(df_split)
        for i, (_, row) in enumerate(df_split.iterrows()):
            img_path = os.path.join(DATASET_ROOT, str(row["class"]), str(row["Image_ID"]))
            with Image.open(img_path) as img:
                img = img.convert("RGB").resize((224, 224), Image.Resampling.BILINEAR)
                arr = (np.array(img, dtype=np.float32) / 255.0 - MEAN) / STD
                arr = np.expand_dims(arr.transpose(2, 0, 1), axis=0).astype(np.float32)
                feat = session.run([out_name], {inp_name: arr})[0].flatten()
                row_dict = {"Image_ID": row["Image_ID"], "class": row["class"]}
                for j in range(576):
                    row_dict[f"feat_{j}"] = float(feat[j])
                rows.append(row_dict)
            if (i + 1) % 500 == 0 or (i + 1) == total:
                print(f"  Processed {i + 1}/{total} images...")
        return pd.DataFrame(rows)

    print("Extracting features for Training Set...")
    train_features_df = extract_features(df_train)
    print("Extracting features for Validation Set...")
    val_features_df = extract_features(df_val)

    train_features_df.to_csv(TRAIN_FEATURE_PATH, index=False)
    val_features_df.to_csv(VAL_FEATURE_PATH, index=False)
    print(f"Saved features to:\n  {TRAIN_FEATURE_PATH}\n  {VAL_FEATURE_PATH}")

print()
print(train_features_df.head())


EXTRACTING FEATURES VIA LOCAL MOBILENETV3 ONNX BACKBONE
Loading ONNX Backbone: c:\Users\priya\Downloads\ok\TinyBayes-App\app\src\main\assets\models\mobilenet_v3_small_features.onnx
Extracting features for Training Set...
  Processed 500/4418 images...
  Processed 1000/4418 images...
  Processed 1500/4418 images...
  Processed 2000/4418 images...
  Processed 2500/4418 images...
  Processed 3000/4418 images...
  Processed 3500/4418 images...
  Processed 4000/4418 images...
  Processed 4418/4418 images...
Extracting features for Validation Set...
  Processed 500/1105 images...
  Processed 1000/1105 images...
  Processed 1105/1105 images...
Saved features to:
  c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\cocoa\run\train_features.csv
  c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\cocoa\run\validation_features.csv

         Image_ID        class    feat_0    feat_1    feat_2    feat_3  \
0   ID_JlL43G.JPG      healthy  0.493657  0.012090  0.015693  0.624960   
1  ID_LYQ46X

In [7]:
# ============================================================
# CELL 4
# TRAIN JACOBI-DMR
# ============================================================

import time

time_jacobi_start = time.time()

# ------------------------------------------------------------
# Separate features and labels
# ------------------------------------------------------------

feat_cols = [c for c in train_features_df.columns if c.startswith("feat_")]

X_train = train_features_df[feat_cols].values.astype(np.float32)
y_train = train_features_df["class"].values

X_val = val_features_df[feat_cols].values.astype(np.float32)
y_val = val_features_df["class"].values

# ------------------------------------------------------------
# One-hot encode targets
# ------------------------------------------------------------

y_train_df = pd.DataFrame({"c": y_train})
y_one_hot = pd.get_dummies(y_train_df["c"])

for cls in CLASS_NAMES:
    if cls not in y_one_hot.columns:
        y_one_hot[cls] = 0

y_one_hot = y_one_hot[CLASS_NAMES]

# ------------------------------------------------------------
# Jacobi regularized solve
# ------------------------------------------------------------

N = len(X_train)
a = 1.0 / N
b = 1.0 / N
k = 1.0

identity = np.eye(X_train.shape[1], dtype=np.float32)
regularization_lambda = 1.0

XtX = np.matmul(X_train.T, X_train)
XtX_reg = XtX + regularization_lambda * identity

jacobi_coefficients = {}

for cls in CLASS_NAMES:
    y_c = y_one_hot[cls].values
    eta = np.log((y_c + a) / (1.0 + k * b))
    Xty = np.matmul(X_train.T, eta)
    beta = np.linalg.solve(XtX_reg, Xty)
    jacobi_coefficients[cls] = beta

# ------------------------------------------------------------
# Validation inference
# ------------------------------------------------------------

preds_jacobi = []
for i in range(len(X_val)):
    x = X_val[i]
    scores = {cls: float(np.dot(x, jacobi_coefficients[cls])) for cls in CLASS_NAMES}
    preds_jacobi.append(max(scores, key=scores.get))

time_jacobi = time.time() - time_jacobi_start
acc_jacobi = accuracy_score(y_val, preds_jacobi)

print()
print("="*60)
print("JACOBI-DMR")
print("="*60)
print(f"Accuracy: {acc_jacobi:.4f}")
print(f"Time: {time_jacobi:.4f}s")
print()
print(classification_report(y_val, preds_jacobi, labels=CLASS_NAMES))
print()
print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_jacobi, labels=CLASS_NAMES))

# ------------------------------------------------------------
# Save coefficients
# ------------------------------------------------------------

JACOBI_JSON_PATH = os.path.join(SAVE_DIR, "jacobi_coefficients.json")

save_dict = {cls: [float(v) for v in jacobi_coefficients[cls]] for cls in CLASS_NAMES}

with open(JACOBI_JSON_PATH, "w") as f:
    json.dump(save_dict, f, indent=4)

print()
print("Jacobi coefficients saved to:")
print(JACOBI_JSON_PATH)



JACOBI-DMR
Accuracy: 0.8271
Time: 0.7021s

              precision    recall  f1-score   support

 anthracnose       0.81      0.80      0.81       313
       cssvd       0.83      0.86      0.85       448
     healthy       0.83      0.81      0.82       344

    accuracy                           0.83      1105
   macro avg       0.83      0.82      0.82      1105
weighted avg       0.83      0.83      0.83      1105


Confusion Matrix:

[[250  35  28]
 [ 35 384  29]
 [ 23  41 280]]

Jacobi coefficients saved to:
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\cocoa\run\jacobi_coefficients.json


In [8]:
# ============================================================
# RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier

import time

time_rf_start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

preds_rf = rf_model.predict(X_val)

time_rf = time.time() - time_rf_start

acc_rf = accuracy_score(y_val, preds_rf)

print()
print("="*60)
print("RANDOM FOREST")
print("="*60)

print(f"Accuracy: {acc_rf:.4f}")
print(f"Time: {time_rf:.4f}s")
print()

print(classification_report(y_val, preds_rf, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_rf, labels=CLASS_NAMES))



RANDOM FOREST
Accuracy: 0.7937
Time: 13.6803s

              precision    recall  f1-score   support

 anthracnose       0.83      0.71      0.76       313
       cssvd       0.77      0.87      0.82       448
     healthy       0.80      0.78      0.79       344

    accuracy                           0.79      1105
   macro avg       0.80      0.78      0.79      1105
weighted avg       0.80      0.79      0.79      1105


Confusion Matrix:

[[221  60  32]
 [ 24 388  36]
 [ 20  56 268]]


In [9]:
# ============================================================
# SUPPORT VECTOR MACHINE (SVM)
# ============================================================

from sklearn.svm import SVC

import time

time_svm_start = time.time()

svm_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    random_state=42
)

svm_model.fit(X_train, y_train)

preds_svm = svm_model.predict(X_val)

time_svm = time.time() - time_svm_start

acc_svm = accuracy_score(y_val, preds_svm)

print()
print("="*60)
print("SUPPORT VECTOR MACHINE")
print("="*60)

print(f"Accuracy: {acc_svm:.4f}")
print(f"Time: {time_svm:.4f}s")
print()

print(classification_report(y_val, preds_svm, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_svm, labels=CLASS_NAMES))



SUPPORT VECTOR MACHINE
Accuracy: 0.8471
Time: 9.0338s

              precision    recall  f1-score   support

 anthracnose       0.83      0.81      0.82       313
       cssvd       0.84      0.87      0.86       448
     healthy       0.87      0.85      0.86       344

    accuracy                           0.85      1105
   macro avg       0.85      0.84      0.85      1105
weighted avg       0.85      0.85      0.85      1105


Confusion Matrix:

[[255  38  20]
 [ 34 389  25]
 [ 18  34 292]]


In [10]:
# ============================================================
# RIDGE CLASSIFIER
# ============================================================

from sklearn.linear_model import RidgeClassifier

import time

time_ridge_start = time.time()

ridge_model = RidgeClassifier(
    alpha=1.0,
    random_state=42
)

ridge_model.fit(X_train, y_train)

preds_ridge = ridge_model.predict(X_val)

time_ridge = time.time() - time_ridge_start

acc_ridge = accuracy_score(y_val, preds_ridge)

print()
print("="*60)
print("RIDGE CLASSIFIER")
print("="*60)

print(f"Accuracy: {acc_ridge:.4f}")
print(f"Time: {time_ridge:.4f}s")
print()

print(classification_report(y_val, preds_ridge, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_ridge, labels=CLASS_NAMES))



RIDGE CLASSIFIER
Accuracy: 0.8253
Time: 0.2044s

              precision    recall  f1-score   support

 anthracnose       0.81      0.79      0.80       313
       cssvd       0.83      0.86      0.84       448
     healthy       0.83      0.81      0.82       344

    accuracy                           0.83      1105
   macro avg       0.82      0.82      0.82      1105
weighted avg       0.83      0.83      0.83      1105


Confusion Matrix:

[[248  38  27]
 [ 35 384  29]
 [ 23  41 280]]


In [11]:
# ============================================================
# LOGISTIC REGRESSION (L1 / LASSO)
# ============================================================

from sklearn.linear_model import LogisticRegression

import time

time_lr_start = time.time()

lr_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train, y_train)

preds_lr = lr_model.predict(X_val)

time_lr = time.time() - time_lr_start

acc_lr = accuracy_score(y_val, preds_lr)

print()
print("="*60)
print("LOGISTIC REGRESSION (L1 / LASSO)")
print("="*60)

print(f"Accuracy: {acc_lr:.4f}")
print(f"Time: {time_lr:.4f}s")
print()

print(classification_report(y_val, preds_lr, labels=CLASS_NAMES))
print()

print("Confusion Matrix:")
print()
print(confusion_matrix(y_val, preds_lr, labels=CLASS_NAMES))



LOGISTIC REGRESSION (L1 / LASSO)
Accuracy: 0.8235
Time: 14.1002s

              precision    recall  f1-score   support

 anthracnose       0.80      0.81      0.81       313
       cssvd       0.82      0.85      0.83       448
     healthy       0.85      0.80      0.82       344

    accuracy                           0.82      1105
   macro avg       0.82      0.82      0.82      1105
weighted avg       0.82      0.82      0.82      1105


Confusion Matrix:

[[253  40  20]
 [ 38 382  28]
 [ 24  45 275]]


In [12]:
# ============================================================
# FINAL MODEL COMPARISON
# ============================================================

comparison_df = pd.DataFrame({
    "Model": [
        "Support Vector Machine",
        "Ridge Classifier",
        "Logistic Regression (L1)",
        "Jacobi-DMR",
        "Random Forest"
    ],
    "Accuracy": [
        acc_svm,
        acc_ridge,
        acc_lr,
        acc_jacobi,
        acc_rf
    ],
    "Execution Time (s)": [
        time_svm,
        time_ridge,
        time_lr,
        time_jacobi,
        time_rf
    ]
})

comparison_df = comparison_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

print()
print("="*70)
print("MODEL COMPARISON")
print("="*70)

print(comparison_df.to_string(index=False))

comparison_path = os.path.join(
    SAVE_DIR,
    "model_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print()
print(f"Comparison saved to:\n{comparison_path}")



MODEL COMPARISON
                   Model  Accuracy  Execution Time (s)
  Support Vector Machine  0.847059            9.033751
              Jacobi-DMR  0.827149            0.702109
        Ridge Classifier  0.825339            0.204400
Logistic Regression (L1)  0.823529           14.100179
           Random Forest  0.793665           13.680272

Comparison saved to:
c:\Users\priya\Downloads\ok\TinyBayes-ML\data\dataset\cocoa\run\model_comparison.csv
